# Module 11 · Gene set enrichment

GSEApy prerank over the limma tables module 10 wrote.

Reads results from disk — no model refitting, no shared kernel with module 10.
Which contrasts exist and what they are called comes from those files, so this
notebook inherits `AXIS` without needing to know about it.

| Section | |
|---|---|
| 01-02 | config, rankings |
| 03 | axis-depletion check |
| 04 | prerank |
| 05 | collapse redundant pathways |
| 06 | ribosome-stripped vs unstripped |
| 07-08 | dot plots, NES grids |
| 09 | functional annotation of axis-unique genes |
| 10 | Hallmark selection |
| 11 | Panel K |

**Databases.** Reactome, KEGG and Hallmark.

**The ribosome question is a parameter, not a filter.** Translational and
ribosomal terms dominate many rankings and crowd everything else out. Stripping
them is defensible but it changes the answer — so both are run, and section 06
asks specifically whether the DNA damage response returns once they are gone. A
filter that changes a conclusion should be visible as a choice.

> **Two languages.** The notebook is Python; the four ggplot dot-plot cells run
> through rpy2 `%%R`, because the house plotting style is R. They read tables
> from disk, so nothing crosses the boundary.

---
## 01 · Config

**Why.** Paths from the environment; each GSEA cell declares the contrast set it
reads. `_env()` is the only line not in the source.

In [ ]:
# -----------------------------------------------------------------------------
# Path resolution - the only addition to the source
# -----------------------------------------------------------------------------
import os
from pathlib import Path


def _env(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(
            f"{name} is not set. Copy .env.example to .env, edit the paths, "
            f"and source it before starting the kernel.")
    return v.rstrip("/")


SEN_DATA = _env("SENESCENCE_DATA")
SEN_REF  = _env("SENESCENCE_REF")
print(f"  SENESCENCE_DATA -> {SEN_DATA}")

# rpy2 bridge for the R plotting cells in sections 07-08
%load_ext rpy2.ipython

---
## 02 · Build the rankings

**Why.** Signed p-value per gene per contrast, `-log10(p) * sign(logFC)`, from the limma moderated-t tables. Ordering on evidence rather than effect size is what keeps a lowly-expressed gene with a large noisy fold change out of the head of the list, where it would otherwise drive the enrichment.

In [ ]:
# ============================================================
# GSEA PRERANK — two contrasts (senescence axis, DAM axis)
# ============================================================
import pandas as pd, numpy as np, gseapy as gp
from pathlib import Path

BASE = Path(file.path(SCRATCH, "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results"))
OUT  = Path(file.path(SCRATCH, "brain/module_06_dge/disease/AD/psychad_ad/Microglia/gsea_prerank"))
for d in [OUT]: d.mkdir(parents=True, exist_ok=True)

COL_GENE, COL_LFC, COL_PVAL, COL_FDR = 'gene','logFC','P.Value','adj.P.Val'  # limma output cols
RANKING_METHOD = 'signed_pval'; FDR_THRESHOLD = 0.05

CONTRASTS = {
    'senescence_axis': 'pseudobulk_DE_SenHiDAMlo_vs_SenHiDAMlo'.replace('SenHiDAMlo_vs_SenHiDAMlo','SenHiDAMlo_vs_SenLoDAMlo') + '_microglia.csv',
    'DAM_axis':        'pseudobulk_DE_SenHiDAMhi_vs_SenHiDAMlo_microglia.csv',
}

def create_ranking(df, method='signed_pval'):
    if method == 'signed_pval':
        pv = df[COL_PVAL].clip(lower=1e-300)
        score = -np.log10(pv) * np.sign(df[COL_LFC])
        r = pd.Series(score.values, index=df[COL_GENE])
    else:
        r = df.set_index(COL_GENE)[COL_LFC]
    return r.replace([np.inf,-np.inf], np.nan).dropna().sort_values(ascending=False)

rankings = {}
for name, fn in CONTRASTS.items():
    df = pd.read_csv(BASE / fn)
    # handle column-name variants
    cols = {c.lower(): c for c in df.columns}
    g = cols.get('gene','gene')
    lfc = cols.get('logfc', cols.get('avg_log2fc'))
    pv  = cols.get('p.value', cols.get('p_val', cols.get('pvalue')))
    rk = (-np.log10(df[pv].clip(lower=1e-300)) * np.sign(df[lfc]))
    rk = pd.Series(rk.values, index=df[g]).replace([np.inf,-np.inf],np.nan).dropna().sort_values(ascending=False)
    rankings[name] = rk
    print(f"{name}: {len(rk):,} genes ranked")
    print(f"  top: {list(rk.head(5).index)}")
    print(f"  bottom: {list(rk.tail(5).index)}\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GSEA STEP 1 — rankings: signed_pval = -log10(p)*sign(logFC), ribosome-stripped
# ════════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np, re
from pathlib import Path
BASE = Path(file.path(SCRATCH, "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results"))
OUT  = Path(file.path(SCRATCH, "brain/module_06_dge/disease/AD/psychad_ad/Microglia/gsea_prerank"))
OUT.mkdir(parents=True, exist_ok=True)

CONTRASTS = {
    'DAMaxis_SnCpos': dict(file='pseudobulk_DE_DAMaxis_SnCpos_microglia.csv', pos='SnC+DAM+', neg='SnC+DAM-'),
    'DAMaxis_SnCneg': dict(file='pseudobulk_DE_DAMaxis_SnCneg_microglia.csv', pos='SnC-DAM+', neg='SnC-DAM-'),
    'SenAxis_DAMpos': dict(file='pseudobulk_DE_SenAxis_DAMpos_microglia.csv', pos='SnC+DAM+', neg='SnC-DAM+'),
    'SenAxis_DAMneg': dict(file='pseudobulk_DE_SenAxis_DAMneg_microglia.csv', pos='SnC+DAM-', neg='SnC-DAM-'),
}
COL_GENE, COL_LFC, COL_PVAL = 'gene', 'logFC', 'P.Value'

RIBO_PAT = re.compile(r'^(RPL|RPS|MRPL|MRPS|RPLP\d|RPSA|FAU)', re.I)
EXTRA = {'RPSA','FAU','RACK1','UBA52'}
def strip_ribo(rk):
    return rk.loc[[g for g in rk.index if not RIBO_PAT.match(str(g)) and str(g).upper() not in EXTRA]]

print("FILE CHECK:")
for name, cfg in CONTRASTS.items():
    p = BASE / cfg['file']
    if p.exists():
        cols = set(pd.read_csv(p, nrows=1).columns); miss = {COL_GENE,COL_LFC,COL_PVAL}-cols
        print(f"  {'✓' if not miss else '✗'} {name:16s}" + (f" MISSING {miss}" if miss else " ok"))
    else: print(f"  ✗ {name:16s} FILE MISSING: {cfg['file']}")

print("\nRANKINGS:")
rankings = {}
for name, cfg in CONTRASTS.items():
    df = pd.read_csv(BASE / cfg['file'])
    score = -np.log10(df[COL_PVAL].clip(lower=1e-300)) * np.sign(df[COL_LFC])
    rk = (pd.Series(score.values, index=df[COL_GENE])
            .replace([np.inf,-np.inf], np.nan).dropna()
            .groupby(level=0).first().sort_values(ascending=False))
    rk_nr = strip_ribo(rk)
    rankings[name] = rk_nr
    print(f"\n  {name}: {len(rk):,} → {len(rk_nr):,} ({len(rk)-len(rk_nr)} ribosomal stripped)")
    print(f"    top (up in {cfg['pos']}):  {list(rk_nr.head(6).index)}")
    print(f"    bot (up in {cfg['neg']}):  {list(rk_nr.tail(6).index)}")
    rk_nr.to_csv(OUT / f"{name}_noribo_signedpval.rnk", sep='\t', header=False)
print("\n✓ Step 1 — rankings built. Confirm: right biology at top/bottom, NO RPL/RPS in top.")

---
## 03 · Axis-depletion check

**Why.** Are the senescence-axis DEGs *depleted* of activation-panel genes? Depletion is a stronger claim than low overlap — it says the two programmes are not merely distinct but anti-correlated in membership.

In [ ]:
# ============================================================
# CHECK A — are senescence-axis DEGs DAM-depleted?
# CHECK B — ORA on senescence-axis genes (find hidden pathway)
# ============================================================
import pandas as pd, numpy as np, gseapy as gp
from scipy.stats import hypergeom

BASE = file.path(SCRATCH, "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results")

A = pd.read_csv(f'{BASE}/pseudobulk_DE_SenHiDAMlo_vs_SenLoDAMlo_microglia.csv')   # senescence axis
B = pd.read_csv(f'{BASE}/pseudobulk_DE_SenHiDAMhi_vs_SenHiDAMlo_microglia.csv')   # DAM axis
for d in (A,B):
    if 'direction' not in d:
        d['direction'] = np.where((d['adj.P.Val']<0.05)&(d['logFC']>0),'Up',
                          np.where((d['adj.P.Val']<0.05)&(d['logFC']<0),'Down','NS'))

A_sig = set(A.loc[A.direction!='NS','gene'].str.upper())
A_up  = set(A.loc[A.direction=='Up','gene'].str.upper())
A_uni = A_sig - set(B.loc[B.direction!='NS','gene'].str.upper())   # senescence-axis UNIQUE
universe = set(A['gene'].str.upper())

# ---- CHECK A: DAM-gene depletion among senescence-axis DEGs ----
# DAM program = the DAM-axis UP genes (the empirical DAM signature)
DAM_prog = set(B.loc[B.direction=='Up','gene'].str.upper()) & universe
print("="*60); print("CHECK A — DAM-gene content of senescence-axis DEGs"); print("="*60)
for label, hits in [("sig (any dir)", A_sig), ("up only", A_up), ("unique", A_uni)]:
    hits = hits & universe
    k = len(hits & DAM_prog); n=len(hits); K=len(DAM_prog); N=len(universe)
    exp = n*K/N
    # enrichment p (over) and depletion p (under)
    p_over  = hypergeom.sf(k-1, N, K, n)
    p_under = hypergeom.cdf(k, N, K, n)
    print(f"  {label:14s}: {k}/{n} are DAM-program genes | exp {exp:.1f} | "
          f"fold {k/exp if exp>0 else 0:.2f} | p_enrich {p_over:.2g} | p_deplete {p_under:.2g}")
print(f"\n  (DAM program = {len(DAM_prog)} DAM-axis up-genes in universe)")

# ---- CHECK B: ORA on senescence-axis gene lists ----
LIBS = ['Reactome_2022','KEGG_2021_Human','GO_Biological_Process_2023']
print("="*60); print("CHECK B (fixed) — ORA on senescence-axis genes"); print("="*60)
for label, glist in [("UNIQUE (102)", sorted(A_uni)), ("ALL sig (164)", sorted(A_sig))]:
    print(f"\n--- {label} ---")
    try:
        enr = gp.enrichr(gene_list=list(glist), gene_sets=LIBS,
                         background=sorted(universe), outdir=None)
        r = enr.results
        if r is None or len(r)==0:
            print("  enrichr returned no results"); continue
        sig = r[r['Adjusted P-value'] < 0.10].sort_values('Adjusted P-value')
        if len(sig)==0:
            # show top 5 raw even if not FDR-sig, to see what's closest
            top = r.sort_values('Adjusted P-value').head(5)
            print("  no terms at FDR<0.10. Closest (raw):")
            for _,row in top.iterrows():
                ov = row.get('Overlap','?')
                print(f"    FDR {row['Adjusted P-value']:.2g}  [{str(row['Gene_set'])[:12]}]  {str(row['Term'])[:48]}  ({ov})")
        else:
            for _,row in sig.head(20).iterrows():
                ov = row.get('Overlap','?')
                print(f"  FDR {row['Adjusted P-value']:.2g}  [{str(row['Gene_set'])[:12]}]  {str(row['Term'])[:50]}  ({ov})")
    except Exception as e:
        print(f"  enrichr error: {type(e).__name__}: {e}")

---
## 04 · Prerank

**Why.** Rankings in, enrichment out, per contrast per database.

In [ ]:
# ============================================================
# Step 2 — GSEA prerank: 2 contrasts × {Reactome, KEGG}
# ============================================================
DATABASES = ['Reactome_2022', 'KEGG_2021_Human']

all_gsea = {}
for name, ranking in rankings.items():
    for db in DATABASES:
        print(f"\n{'='*60}\n{name} | {db}\n{'='*60}")
        outdir = OUT / name / db
        try:
            pre = gp.prerank(
                rnk=ranking,
                gene_sets=db,
                outdir=str(outdir),
                min_size=15, max_size=500,
                permutation_num=1000,
                threads=4, seed=42, verbose=False
            )
            res = pre.res2d.copy()
            res['NES']        = pd.to_numeric(res['NES'], errors='coerce')
            res['FDR q-val']  = pd.to_numeric(res['FDR q-val'], errors='coerce')
            res['NOM p-val']  = pd.to_numeric(res['NOM p-val'], errors='coerce')
            res['contrast'] = name; res['Database'] = db
            all_gsea[(name, db)] = res

            n_sig  = (res['FDR q-val'] < FDR_THRESHOLD).sum()
            n_up   = ((res['FDR q-val'] < FDR_THRESHOLD) & (res['NES'] > 0)).sum()
            n_down = ((res['FDR q-val'] < FDR_THRESHOLD) & (res['NES'] < 0)).sum()
            print(f"  ✓ {len(res)} pathways tested | sig {n_sig} (up {n_up}, down {n_down})")
        except Exception as e:
            print(f"  ✗ FAILED: {type(e).__name__}: {e}")

# combine + save
if all_gsea:
    combined = pd.concat(all_gsea.values(), ignore_index=True)
    outfile = OUT / 'gsea_prerank_both_contrasts_microglia.csv'
    combined.to_csv(outfile, index=False)
    print(f"\n✓ Saved: {outfile}  ({len(combined)} rows)")

In [ ]:
# ============================================================
# Step 3 — top significant pathways per contrast (Reactome + KEGG)
# ============================================================
import pandas as pd
combined = pd.concat(all_gsea.values(), ignore_index=True)
TERMCOL = 'Term'

def show(name, db, n=15):
    r = combined[(combined['contrast']==name) & (combined['Database']==db)].copy()
    sig = r[r['FDR q-val'] < 0.05].sort_values('NES', ascending=False)
    print(f"\n{'='*70}\n{name} | {db} — {len(sig)} sig (showing top {min(n,len(sig))} by NES)\n{'='*70}")
    if len(sig)==0:
        print("  (none)"); return
    for _, row in sig.head(n).iterrows():
        term = str(row[TERMCOL]).split('__')[-1][:60]   # strip db prefix if present
        print(f"  NES {row['NES']:+.2f}  FDR {row['FDR q-val']:.1e}  {term}")

for name in ['senescence_axis','DAM_axis']:
    for db in ['Reactome_2022','KEGG_2021_Human']:
        show(name, db)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GSEA PRERANK — Step 1: CONFIG + FILE CHECK  (all four 2×2 contrasts)
# ════════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np
from pathlib import Path

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG, X_TAG = 'SnC', 'IRM'          # senescence axis, activation axis
BASE = Path(file.path(SCRATCH, "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results"))
OUT  = Path(file.path(SCRATCH, "brain/module_06_dge/disease/AD/psychad_ad/Microglia/gsea_prerank"))
COL  = dict(gene='gene', lfc='logFC', pval='P.Value', fdr='adj.P.Val', t='t')
# ════════════════════════════════════════════════════════════════════════════
OUT.mkdir(parents=True, exist_ok=True)
yp, yn, xp, xn = f'{Y_TAG}+', f'{Y_TAG}-', f'{X_TAG}+', f'{X_TAG}-'

# each contrast: file + the two sides (POS side = NES>0 = up in TEST/first quadrant)
# circular flag: senescence-axis contrasts split on senescence_score → defining genes expected
CONTRASTS = {
    f'{X_TAG}axis_{Y_TAG}pos': dict(file=f'pseudobulk_DE_{X_TAG}axis_{Y_TAG}pos_microglia.csv',
                           pos=f'{yp}{xp}', neg=f'{yp}{xn}', axis=f'{X_TAG} axis | senescent',       circular=False),
    f'{X_TAG}axis_{Y_TAG}neg': dict(file=f'pseudobulk_DE_{X_TAG}axis_{Y_TAG}neg_microglia.csv',
                           pos=f'{yn}{xp}', neg=f'{yn}{xn}', axis=f'{X_TAG} axis | non-senescent',   circular=False),
    f'{Y_TAG}axis_{X_TAG}pos': dict(file=f'pseudobulk_DE_{Y_TAG}axis_{X_TAG}pos_microglia.csv',
                           pos=f'{yp}{xp}', neg=f'{yn}{xp}', axis=f'Sen axis | {xp}',     circular=True),
    f'{Y_TAG}axis_{X_TAG}neg': dict(file=f'pseudobulk_DE_{Y_TAG}axis_{X_TAG}neg_microglia.csv',
                           pos=f'{yp}{xn}', neg=f'{yn}{xn}', axis=f'Sen axis | {xn}',     circular=True),
}

print("="*64); print("FILE + COLUMN CHECK"); print("="*64)
print(f"BASE exists: {BASE.exists()}\n")
ok = {}
for name, cfg in CONTRASTS.items():
    p = BASE / cfg['file']
    if not p.exists():
        print(f"✗ {name:18s} MISSING — {cfg['file']}"); ok[name] = False; continue
    df = pd.read_csv(p); cols = set(df.columns)
    miss = {COL['gene'], COL['lfc'], COL['t']} - cols
    flag = "  [circular]" if cfg['circular'] else ""
    print(f"{'✓' if not miss else '✗'} {name:18s} {len(df):>6,} rows | has 't': {COL['t'] in cols} | "
          f"+NES = up in {cfg['pos']}{flag}")
    if miss: print(f"    MISSING COLS: {miss} | have: {list(df.columns)}")
    ok[name] = not miss
print("\nready:", [k for k, v in ok.items() if v]); print("="*64)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GSEA PRERANK — Step 2: build rankings (limma moderated-t), all contrasts
# ════════════════════════════════════════════════════════════════════════════
RANK_BY = 't'
rankings = {}
for name, cfg in CONTRASTS.items():
    if not ok.get(name): continue
    df = pd.read_csv(BASE / cfg['file'])
    r = (df.dropna(subset=[RANK_BY, COL['gene']])
           .drop_duplicates(subset=COL['gene'])
           .set_index(COL['gene'])[RANK_BY]
           .replace([np.inf,-np.inf], np.nan).dropna()
           .sort_values(ascending=False))
    rankings[name] = r
    print(f"{name}: {len(r):,} genes ranked")
    print(f"  top (up in {cfg['pos']}):  {list(r.head(6).index)}")
    print(f"  bot (up in {cfg['neg']}):  {list(r.tail(6).index)}")
    r.to_csv(OUT / f"{name}.rnk", sep='\t', header=False)
    print()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GSEA PRERANK — Step 3: run prerank per contrast (Reactome + Hallmark)
#   needs internet on the node; else point me at a local .gmt
# ════════════════════════════════════════════════════════════════════════════
import gseapy as gp
GENE_SETS = ['MSigDB_Hallmark_2020', 'Reactome_2022']
MIN_SIZE, MAX_SIZE, NPERM = 15, 500, 1000
gsea_res = {}
for name, r in rankings.items():
    cfg = CONTRASTS[name]
    rnk = r.reset_index(); rnk.columns = ['gene','score']
    pre = gp.prerank(rnk=rnk, gene_sets=GENE_SETS,
                     min_size=MIN_SIZE, max_size=MAX_SIZE,
                     permutation_num=NPERM, seed=1,
                     outdir=str(OUT / name), no_plot=True, threads=4)
    df = pre.res2d.copy()
    df['FDR q-val'] = pd.to_numeric(df['FDR q-val'], errors='coerce')
    df['NES']       = pd.to_numeric(df['NES'], errors='coerce')
    gsea_res[name] = df.sort_values('FDR q-val')
    sig = df[df['FDR q-val']<0.05]
    print(f"\n{'='*60}\n{name}  ({cfg['axis']})  —  {len(sig)} pathways FDR<0.05")
    print(f"  ── up in {cfg['pos']} (NES>0) ──")
    print(sig[sig.NES>0].sort_values('NES',ascending=False)[['Term','NES','FDR q-val']].head(8).to_string(index=False))
    print(f"  ── up in {cfg['neg']} (NES<0) ──")
    print(sig[sig.NES<0].sort_values('NES')[['Term','NES','FDR q-val']].head(8).to_string(index=False))
    df.to_csv(OUT / f"gsea_{name}.csv", index=False)

---
## 05 · Collapse redundant pathways

**Why.** Reactome and Hallmark contain nested sets sharing most of their genes, so one signal surfaces as a dozen apparently independent significant pathways. Collapsing on leading-edge Jaccard reports it once.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GSEA PRERANK — Step 4: collapse redundant pathways (leading-edge Jaccard)
# ════════════════════════════════════════════════════════════════════════════
def collapse_redundant(df, jaccard_cut=0.5, le_col='Lead_genes'):
    df = df.sort_values('FDR q-val').copy()
    le = {row['Term']: set(str(row[le_col]).split(';')) for _,row in df.iterrows()}
    kept, dropped = [], set()
    for term in df['Term']:
        if term in dropped: continue
        kept.append(term)
        for other in df['Term']:
            if other==term or other in dropped: continue
            a,b = le[term], le[other]
            if a and b and len(a&b)/len(a|b) >= jaccard_cut:
                dropped.add(other)
    return df[df['Term'].isin(kept)]

gsea_collapsed = {}
for name, df in gsea_res.items():
    cfg = CONTRASTS[name]
    sig = df[df['FDR q-val']<0.05].copy()
    col = collapse_redundant(sig, jaccard_cut=0.5)
    gsea_collapsed[name] = col
    print(f"\n{name}: {len(sig)} sig → {len(col)} after collapse")
    print(f"  {cfg['pos']:10s}:", col[col.NES>0].sort_values('NES',ascending=False)['Term'].head(6).tolist())
    print(f"  {cfg['neg']:10s}:", col[col.NES<0].sort_values('NES')['Term'].head(6).tolist())
    col.to_csv(OUT / f"gsea_{name}_collapsed.csv", index=False)

---
## 06 · Ribosome-stripped versus unstripped

**Why.** The comparison the parameter exists for. One pass strips ribosomal genes from the ranking; the diagnostic then asks whether DDR terms return once they are gone — a directly interpretable answer to whether translation was masking the damage response.

In [ ]:
# ============================================================
# Step 2b — GSEA prerank, RIBOSOMAL GENES STRIPPED from ranking
# ============================================================
import re

# ribosomal + obvious translation-block genes to remove from the RANKING
# (RPL*, RPS*, MRPL*, MRPS*, RPLP*, RPSA, FAU, plus EEF/EIF elongation/initiation)
RIBO_PAT = re.compile(r'^(RPL|RPS|MRPL|MRPS|RPLP\d|RPSA|FAU)', re.I)
EXTRA = {'RPSA','FAU','RACK1','UBA52'}   # ribosomal genes not matching the prefix

def strip_ribo(rk):
    keep = [g for g in rk.index if not RIBO_PAT.match(str(g)) and str(g).upper() not in EXTRA]
    return rk.loc[keep]

DATABASES = ['Reactome_2022', 'KEGG_2021_Human']
all_gsea_nr = {}

for name, ranking in rankings.items():
    rk = strip_ribo(ranking)
    print(f"\n{name}: {len(ranking):,} → {len(rk):,} genes after stripping ribosomal "
          f"({len(ranking)-len(rk)} removed)")
    print(f"  new top: {list(rk.head(6).index)}")
    for db in DATABASES:
        outdir = OUT / f"{name}_noribo" / db
        try:
            pre = gp.prerank(rnk=rk, gene_sets=db, outdir=str(outdir),
                             min_size=15, max_size=500, permutation_num=1000,
                             threads=4, seed=42, verbose=False)
            res = pre.res2d.copy()
            res['NES']       = pd.to_numeric(res['NES'], errors='coerce')
            res['FDR q-val'] = pd.to_numeric(res['FDR q-val'], errors='coerce')
            res['contrast']=name; res['Database']=db
            all_gsea_nr[(name,db)] = res
            n_sig=(res['FDR q-val']<0.05).sum()
            n_up=((res['FDR q-val']<0.05)&(res['NES']>0)).sum()
            n_dn=((res['FDR q-val']<0.05)&(res['NES']<0)).sum()
            print(f"  {db}: {len(res)} tested | sig {n_sig} (up {n_up}, down {n_dn})")
        except Exception as e:
            print(f"  {db} FAILED: {e}")

combined_nr = pd.concat(all_gsea_nr.values(), ignore_index=True)
combined_nr.to_csv(OUT / 'gsea_prerank_noribo_both_contrasts_microglia.csv', index=False)
print(f"\n✓ saved noribo results ({len(combined_nr)} rows)")

# show top per contrast
def show(name, db, n=15):
    r = combined_nr[(combined_nr['contrast']==name)&(combined_nr['Database']==db)]
    sig = r[r['FDR q-val']<0.05].sort_values('NES', ascending=False)
    print(f"\n{'='*68}\n{name} | {db} — {len(sig)} sig (top {min(n,len(sig))} by NES)\n{'='*68}")
    if len(sig)==0: print("  (none)"); return
    for _,row in sig.head(n).iterrows():
        print(f"  NES {row['NES']:+.2f}  FDR {row['FDR q-val']:.1e}  {str(row['Term'])[:62]}")

for name in ['senescence_axis','DAM_axis']:
    for db in ['Reactome_2022','KEGG_2021_Human']:
        show(name, db)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC — Sen-axis GSEA WITHOUT ribosome stripping (does DDR return?)
# ════════════════════════════════════════════════════════════════════════════
import gseapy as gp
# rebuild FULL (unstripped) signed_pval rankings for the two Sen contrasts
rankings_full = {}
for name in ['SenAxis_DAMpos','SenAxis_DAMneg']:
    cfg = CONTRASTS[name]; df = pd.read_csv(BASE / cfg['file'])
    score = -np.log10(df[COL_PVAL].clip(lower=1e-300)) * np.sign(df[COL_LFC])
    rankings_full[name] = (pd.Series(score.values, index=df[COL_GENE])
                           .replace([np.inf,-np.inf],np.nan).dropna()
                           .groupby(level=0).first().sort_values(ascending=False))

for name in ['SenAxis_DAMpos','SenAxis_DAMneg']:
    cfg=CONTRASTS[name]
    rnk=rankings_full[name].reset_index(); rnk.columns=['gene','score']
    pre=gp.prerank(rnk=rnk, gene_sets=['Reactome_2022'], min_size=15, max_size=500,
                   permutation_num=1000, seed=42, threads=4, outdir=None, no_plot=True, verbose=False)
    d=pre.res2d.copy(); d['NES']=pd.to_numeric(d['NES'],errors='coerce'); d['FDR q-val']=pd.to_numeric(d['FDR q-val'],errors='coerce')
    sig=d[d['FDR q-val']<0.05]
    print(f"\n{name} (UNSTRIPPED) — {len(sig)} sig Reactome")
    for _,r in sig.sort_values('NES',ascending=False).head(15).iterrows():
        term=r['Term'].split('__')[-1]
        print(f"  NES {r['NES']:+.2f}  FDR {r['FDR q-val']:.1e}  {term[:55]}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GSEA — UNSTRIPPED rankings (signed_pval), prerank Reactome + KEGG · all 4
# ════════════════════════════════════════════════════════════════════════════
import gseapy as gp
# unstripped signed_pval rankings for all four
rankings_full = {}
for name, cfg in CONTRASTS.items():
    df = pd.read_csv(BASE / cfg['file'])
    score = -np.log10(df[COL_PVAL].clip(lower=1e-300)) * np.sign(df[COL_LFC])
    rankings_full[name] = (pd.Series(score.values, index=df[COL_GENE])
                           .replace([np.inf,-np.inf],np.nan).dropna()
                           .groupby(level=0).first().sort_values(ascending=False))

DATABASES = ['Reactome_2022', 'KEGG_2021_Human']
gsea_res = {}
for name, rk in rankings_full.items():
    cfg = CONTRASTS[name]
    rnk = rk.reset_index(); rnk.columns = ['gene','score']
    pre = gp.prerank(rnk=rnk, gene_sets=DATABASES, min_size=15, max_size=500,
                     permutation_num=1000, seed=42, threads=4,
                     outdir=str(OUT / f"{name}_full"), no_plot=True, verbose=False)
    df = pre.res2d.copy()
    df['NES'] = pd.to_numeric(df['NES'], errors='coerce')
    df['FDR q-val'] = pd.to_numeric(df['FDR q-val'], errors='coerce')
    gsea_res[name] = df.sort_values('FDR q-val')
    nsig = (df['FDR q-val']<0.05).sum()
    print(f"{name:16s} ({cfg['pos']} vs {cfg['neg']}): {nsig} sig")
    df.to_csv(OUT / f"gsea_{name}_full.csv", index=False)
print("\n✓ prerank done (unstripped) — gsea_res holds Reactome + KEGG")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# LIST — top Reactome pathways per contrast (both directions, unstripped)
# ════════════════════════════════════════════════════════════════════════════
PREFIX='Reactome_2022__'; FDR_SIG=0.05
import re
def clean(t): t=t.split('__',1)[-1]; return re.sub(r'\s*R-HSA-\d+\s*$','',t).strip()

for name, cfg in CONTRASTS.items():
    df=gsea_res[name]; df=df[df['Term'].str.startswith(PREFIX)].copy()
    df['pathway']=df['Term'].map(clean)
    sig=df[df['FDR q-val']<FDR_SIG]
    up=sig[sig.NES>0].sort_values('NES',ascending=False)
    dn=sig[sig.NES<0].sort_values('NES')
    print(f"\n{'='*66}\n{name}  ({cfg['pos']} vs {cfg['neg']}) — {len(sig)} sig\n{'='*66}")
    print(f"  ▲ up in {cfg['pos']} ({len(up)}):")
    for _,r in up.head(15).iterrows(): print(f"    {r['NES']:+.2f}  {r['pathway']}")
    print(f"  ▼ up in {cfg['neg']} ({len(dn)}):")
    for _,r in dn.head(15).iterrows(): print(f"    {r['NES']:+.2f}  {r['pathway']}")

---
## 07 · Dot plots

**Why.** Top pathways per contrast, both directions. The ribosomal-lead filter is applied at display rather than at ranking, so the underlying enrichment is unchanged and only the panel is decluttered.

These four cells are R — house plotting style — and run through `%%R`.

In [ ]:
%%R
# minimal house theme — drop-in for theme_clean
theme_clean <- function(base_size = 10) {
    theme_classic(base_size = base_size) +
    theme(
        panel.border   = element_rect(color = "#333333", fill = NA, linewidth = 0.5),
        axis.line      = element_blank(),
        panel.grid.major.x = element_line(color = "grey92", linewidth = 0.3),
        strip.background   = element_blank(),
        plot.title     = element_text(face = "bold"),
        legend.key.size = unit(0.4, "cm")
    )
}

# figure saver — writes PNG (raster preview) + PDF (vector for publication)
FIG_DIR <- file.path(SCRATCH, "brain/module_06_dge/figures")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)

save_figure <- function(plot, name, width = 8, height = 6, dpi = 300, formats = c("png","pdf", "svg")) {
    paths <- character(0)
    for (fmt in formats) {
        fp <- file.path(FIG_DIR, paste0(name, ".", fmt))
        ggsave(fp, plot = plot, width = width, height = height, dpi = dpi,
               bg = "white", device = fmt)
        paths <- c(paths, fp)
    }
    cat("saved:\n"); cat(paste0("  ", paths, collapse = "\n"), "\n")
    invisible(paths)
}

In [ ]:
%%R
# ════════════════════════════════════════════════════════════════════════════
# GSEA DOT PLOT — both contrasts, ribosomal/translation terms filtered out
#   size = -log10(FDR), color = NES, faceted by contrast
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(dplyr); library(ggplot2); library(stringr) })

g <- read.csv(file.path(SCRATCH, "brain/module_06_dge/disease/AD/psychad_ad/Microglia/gsea_prerank/gsea_prerank_both_contrasts_microglia.csv"))

# ── clean term names: strip Reactome R-HSA / KEGG cruft ─────────────────────
clean_term <- function(x) {
    x <- gsub("\\s*R-HSA-[0-9]+\\s*$", "", x)   # Reactome IDs
    x <- gsub("\\s*Homo sapiens.*$", "", x)
    x <- gsub("\\s*\\(.*\\)\\s*$", "", x)
    str_trim(x)
}
g$term_clean <- clean_term(g$Term)

# ── ribosomal/translation artifact filter (same prefixes as ranking strip) ──
RIBO_PAT <- paste(c("ribosom","translation","rRNA","Influenza","Nonsense",
    "Eukaryotic Translation","Peptide Chain","SRP-dependent","Selenoamino",
    "Selenocysteine","GTP Hydrolysis","Formation Of A Pool","Cap-Dependent",
    "Major Pathway Of rRNA","Viral mRNA","40S","60S","EIF2AK4","GCN2",
    "Coronavirus","SARS-CoV","HCMV","Metabolism Of Amino Acids",
    "Response To Starvation","Metabolism Of RNA","L13a","Amino Acid Deficiency",
    "Viral Infection","Infectious disease"), collapse="|")

g$is_ribo <- grepl(RIBO_PAT, g$term_clean, ignore.case=TRUE)

# ── keep significant, non-ribosomal; top N per contrast by FDR ──────────────
FDR_CUT <- 0.05; TOP_N <- 12
gf <- g %>%
    filter(FDR.q.val < FDR_CUT, !is_ribo) %>%
    group_by(contrast) %>%
    arrange(FDR.q.val, desc(abs(NES))) %>%
    slice_head(n = TOP_N) %>%
    ungroup()

cat(sprintf("After ribosomal filter + FDR<%.2f: %d terms (sen %d, DAM %d)\n",
            FDR_CUT, nrow(gf),
            sum(gf$contrast=="senescence_axis"), sum(gf$contrast=="DAM_axis")))

# plotting transforms
gf <- gf %>% mutate(
    neglog_fdr = -log10(pmax(FDR.q.val, 1e-4)),     # cap at 1e-4 so 0-FDR dots aren't infinite
    contrast = factor(contrast, levels=c("senescence_axis","DAM_axis"),
                      labels=c("Senescence axis","DAM axis")),
    term_clean = str_trunc(term_clean, 42))
# order terms within each facet by NES
gf$term_clean <- reorder(gf$term_clean, gf$NES)

p <- ggplot(gf, aes(NES, term_clean)) +
    geom_segment(aes(x=min(gf$NES)-0.1, xend=NES, yend=term_clean),
                 color="grey85", linewidth=0.3) +
    geom_point(aes(size=neglog_fdr, fill=NES), shape=21, color="grey30", stroke=0.3) +
    scale_fill_gradient(low="#FCD2C2", high="#B2182B", name="NES") +
    scale_size_continuous(range=c(2,7), name=expression(-log[10]~FDR)) +
    facet_wrap(~contrast, scales="free_y", ncol=2) +
    labs(title="GSEA enriched pathways: senescence axis vs DAM axis (microglia)",
         subtitle="gseapy prerank, Reactome + KEGG. Ribosomal/translation terms removed (ranking artifact). FDR<0.05, top 12/contrast.",
         x="Normalized Enrichment Score (NES)", y=NULL) +
    theme_clean(base_size=10) +
    theme(plot.subtitle=element_text(size=7, color="grey40"),
          strip.text=element_text(size=10, face="bold"),
          axis.text.y=element_text(size=8),
          panel.spacing=unit(1,"lines"),
          legend.position="right", legend.box="vertical")

options(repr.plot.width=12, repr.plot.height=5.5); print(p)
save_figure(p, "gsea_dotplot_sen_vs_DAM_axis_riboFiltered_microglia", width=12, height=5.5)
cat("\n✓ dot plot done.\n")

In [ ]:
%%R
# ════════════════════════════════════════════════════════════════════════════
# DIAGNOSE shared terms — compare lead genes across the two contrasts
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(dplyr); library(stringr) })

g <- read.csv(file.path(SCRATCH, "brain/module_06_dge/disease/AD/psychad_ad/Microglia/gsea_prerank/gsea_prerank_both_contrasts_microglia.csv"))

clean_term <- function(x) str_trim(gsub("\\s*R-HSA-[0-9]+\\s*$","", gsub("\\s*Homo sapiens.*$","", x)))
g$term_clean <- clean_term(g$Term)

# significant terms per contrast
sig <- g %>% filter(FDR.q.val < 0.05)

# terms appearing in BOTH contrasts
shared_terms <- intersect(
    sig$term_clean[sig$contrast=="senescence_axis"],
    sig$term_clean[sig$contrast=="DAM_axis"])
cat(sprintf("Shared significant terms (both axes): %d\n", length(shared_terms)))
cat(paste(" -", shared_terms), sep="\n"); cat("\n\n")

# for each shared term, compare the lead genes between the two axes
jacc <- function(a,b){ a<-unlist(strsplit(a,";")); b<-unlist(strsplit(b,";"))
    length(intersect(a,b))/length(union(a,b)) }

cat("=", strrep("=",78), "\n", sep="")
cat("LEAD-GENE COMPARISON per shared term (Jaccard of lead genes: sen vs DAM)\n")
cat("=", strrep("=",78), "\n", sep="")
for (t in shared_terms) {
    sg <- sig$Lead_genes[sig$term_clean==t & sig$contrast=="senescence_axis"][1]
    dg <- sig$Lead_genes[sig$term_clean==t & sig$contrast=="DAM_axis"][1]
    j <- jacc(sg, dg)
    sg_v <- unlist(strsplit(sg,";")); dg_v <- unlist(strsplit(dg,";"))
    # are these ribosomal/cytoskeletal?
    ribo <- mean(grepl("^RP[LS]|^MRP|^FAU|^RPLP", union(sg_v,dg_v)))
    cat(sprintf("\n■ %s\n", str_trunc(t,60)))
    cat(sprintf("   lead-gene Jaccard (sen vs DAM): %.2f | %.0f%% ribosomal\n", j, 100*ribo))
    cat(sprintf("   SEN lead: %s\n", str_trunc(sg, 90)))
    cat(sprintf("   DAM lead: %s\n", str_trunc(dg, 90)))
}
cat("\n\nInterpretation guide:\n")
cat("  high Jaccard (>0.5) + ribosomal  -> residual artifact, removable\n")
cat("  high Jaccard, NOT ribosomal      -> same genes -> genuine shared signal (keep)\n")
cat("  low Jaccard                      -> different genes per axis -> NOT really 'shared' (keep, they differ)\n")

In [ ]:
%%R
# ════════════════════════════════════════════════════════════════════════════
# DOT PLOT — ribosomal-lead filter, NO tidytext (manual facet ordering)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(dplyr); library(ggplot2); library(stringr) })

g <- read.csv(file.path(SCRATCH, "brain/module_06_dge/disease/AD/psychad_ad/Microglia/gsea_prerank/gsea_prerank_both_contrasts_microglia.csv"))
clean_term <- function(x) str_trim(gsub("\\s*R-HSA-[0-9]+\\s*$","", gsub("\\s*Homo sapiens.*$","", x)))
g$term_clean <- clean_term(g$Term)

ribo_frac <- function(lead) {
    v <- unlist(strsplit(lead, ";")); if (!length(v)) return(0)
    mean(grepl("^RP[LS]|^RPLP|^MRP[LS]|^FAU$|^RACK1$", v))
}
g$ribo_frac <- sapply(g$Lead_genes, ribo_frac)

# also drop a few explicitly translation/viral-proxy names that aren't RP-gene but are artifact
NAME_DROP <- "^Translation$|HCMV|Influenza|Coronavirus|SARS|Viral mRNA|rRNA|Ribosome|tRNA"

RIBO_CUT <- 0.40; FDR_CUT <- 0.05; TOP_N <- 12
gf <- g %>%
    filter(FDR.q.val < FDR_CUT, ribo_frac <= RIBO_CUT, !grepl(NAME_DROP, term_clean, ignore.case=TRUE)) %>%
    group_by(contrast) %>% arrange(FDR.q.val, desc(abs(NES))) %>%
    slice_head(n=TOP_N) %>% ungroup()

cat(sprintf("Surviving: %d (sen %d, DAM %d)\n", nrow(gf),
            sum(gf$contrast=="senescence_axis"), sum(gf$contrast=="DAM_axis")))

# manual ordering: unique label per facet by prefixing, then relabel
gf <- gf %>% mutate(
    neglog_fdr = -log10(pmax(FDR.q.val, 1e-4)),
    contrast = factor(contrast, levels=c("senescence_axis","DAM_axis"),
                      labels=c("Senescence axis","DAM axis")),
    term_clean = str_trunc(term_clean, 42))
# order rows within facet by NES using a composite factor
gf <- gf %>% arrange(contrast, NES)
gf$row_id <- factor(seq_len(nrow(gf)))                      # unique per row
lab_map <- setNames(gf$term_clean, gf$row_id)               # row_id -> label

p <- ggplot(gf, aes(NES, row_id)) +
    geom_segment(aes(x=min(NES)-0.1, xend=NES, yend=row_id), color="grey85", linewidth=0.3) +
    geom_point(aes(size=neglog_fdr, fill=NES), shape=21, color="grey30", stroke=0.3) +
    scale_y_discrete(labels=lab_map) +
    scale_fill_gradient(low="#FCD2C2", high="#B2182B", name="NES") +
    scale_size_continuous(range=c(2,7), name=expression(-log[10]~FDR)) +
    facet_wrap(~contrast, scales="free_y", ncol=2) +
    labs(title="GSEA enriched pathways: senescence axis vs DAM axis (microglia)",
         subtitle="gseapy prerank. Ribosomal/translation artifact terms removed (lead-gene + name filter). FDR<0.05, top 12/contrast.",
         x="Normalized Enrichment Score (NES)", y=NULL) +
    theme_clean(base_size=10) +
    theme(plot.subtitle=element_text(size=7, color="grey40"),
          strip.text=element_text(size=10, face="bold"),
          axis.text.y=element_text(size=8), panel.spacing=unit(1,"lines"),
          legend.position="right", legend.box="vertical")

options(repr.plot.width=12, repr.plot.height=5.5); print(p)
save_figure(p, "gsea_dotplot_sen_vs_DAM_riboFiltered_clean_microglia", width=12, height=5.5)
cat("\n✓ dot plot done.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GSEA PRERANK — Step 5: dot plot per contrast (top5 each side)
# ════════════════════════════════════════════════════════════════════════════
import textwrap, matplotlib as mpl, matplotlib.pyplot as plt
mpl.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none",
    "font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "font.size":8,"axes.linewidth":0.6})

def clean(t):
    t = t.split('__',1)[-1]; t = t.rsplit(' R-HSA',1)[0]; return t.strip()

def gsea_dotplot(name):
    cfg = CONTRASTS[name]; col = gsea_collapsed[name].copy()
    if col.empty: print(f"{name}: nothing significant — skipping plot"); return
    col['FDR q-val'] = col['FDR q-val'].clip(lower=1e-10)
    col['neglog10FDR'] = -np.log10(col['FDR q-val'])
    col['label'] = col['Term'].map(clean)
    pos = col[col.NES>0].sort_values('NES',ascending=False).head(5)
    neg = col[col.NES<0].sort_values('NES',ascending=True).head(5)
    d = pd.concat([neg.sort_values('NES'), pos.sort_values('NES')]).reset_index(drop=True)
    if d.empty: print(f"{name}: no rows to plot"); return
    d['wrapped'] = d['label'].map(lambda s: "\n".join(textwrap.wrap(s, width=32)))
    y = np.arange(len(d))
    smin, smax = d['neglog10FDR'].min(), d['neglog10FDR'].max()
    ssize = lambda v: 40 + (v-smin)/(smax-smin+1e-9)*170
    vmax = max(abs(d.NES))
    fig, ax = plt.subplots(figsize=(5.8, 0.55*len(d)+1.1), dpi=300)
    sc = ax.scatter(d['NES'], y, c=d['NES'], s=[ssize(v) for v in d['neglog10FDR']],
                    cmap='RdBu_r', vmin=-vmax, vmax=vmax, edgecolor='black', linewidth=0.4, zorder=3)
    ax.axvline(0, color='0.6', lw=0.6, ls='--', zorder=1)
    ax.set_yticks(y); ax.set_yticklabels(d['wrapped'], fontsize=6.8)
    ax.set_xlabel(f"NES   ({cfg['neg']}  \u2190    \u2192  {cfg['pos']})", fontsize=8)
    ax.set_title(f"GSEA: {cfg['axis']}\n{cfg['pos']} vs {cfg['neg']}",
                 fontsize=9, fontweight='bold', loc='left')
    for s in ['top','right']: ax.spines[s].set_visible(False)
    ax.tick_params(labelsize=6.8); ax.set_ylim(-0.6, len(d)-0.4)
    cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("NES", fontsize=7.5); cbar.ax.tick_params(labelsize=6.5)
    for v in np.linspace(smin, smax, 3):
        ax.scatter([], [], s=ssize(v), c='0.6', edgecolor='black', linewidth=0.4, label=f"{v:.1f}")
    ax.legend(title="$-$log$_{10}$ FDR", loc='lower right', fontsize=6.5,
              title_fontsize=6.5, frameon=False, labelspacing=1.2, borderpad=0.6)
    fig.tight_layout()
    for ext in ('pdf','png','svg'):
        fig.savefig(str(OUT / f"gsea_dotplot_{name}_top5.{ext}"), dpi=300, bbox_inches='tight')
    plt.show(); print(f"  saved gsea_dotplot_{name}_top5.{{pdf,png,svg}}")

for name in CONTRASTS:
    if name in gsea_collapsed: gsea_dotplot(name)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# INSPECT — full significant pathway list per contrast (both directions)
# ════════════════════════════════════════════════════════════════════════════
pd.set_option('display.max_rows', None, 'display.width', 200, 'display.max_colwidth', 80)
def clean(t):
    t = t.split('__',1)[-1]; t = t.rsplit(' R-HSA',1)[0]; return t.strip()

for name, cfg in CONTRASTS.items():
    if name not in gsea_res: continue
    sig = gsea_res[name][gsea_res[name]['FDR q-val']<0.05].copy()
    sig['pathway'] = sig['Term'].map(clean)
    print("\n" + "="*70)
    print(f"{name}  ({cfg['axis']})  —  {len(sig)} sig pathways")
    print("="*70)
    up = sig[sig.NES>0].sort_values('NES',ascending=False)
    dn = sig[sig.NES<0].sort_values('NES')
    print(f"\n  ▲ up in {cfg['pos']}  ({len(up)} pathways):")
    print(up[['pathway','NES','FDR q-val']].to_string(index=False))
    print(f"\n  ▼ up in {cfg['neg']}  ({len(dn)} pathways):")
    if len(dn): print(dn[['pathway','NES','FDR q-val']].to_string(index=False))
    else:       print("    (none significant)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GSEA — publication-grade diverging bars (Nature/Cell style)
#   compact · hairline axis · muted diverging palette · full wrapped names
# ════════════════════════════════════════════════════════════════════════════
import textwrap, matplotlib as mpl, matplotlib.pyplot as plt
mpl.rcParams.update({
    "pdf.fonttype":42, "ps.fonttype":42, "svg.fonttype":"none",
    "font.family":"sans-serif", "font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "font.size":8, "axes.linewidth":0.5,
    "xtick.major.width":0.5, "ytick.major.width":0.5,
})

DROP_PAT = ['RUNX1 Interacts With Co-Factors']
N_SIDE   = 6
UP_COL, DN_COL = "#B2182B", "#2166AC"      # muted diverging (RdBu poles)
WRAP     = 46

def clean(t):
    t = t.split('__',1)[-1]; t = t.rsplit(' R-HSA',1)[0]; return t.strip()

def gsea_bars(name):
    cfg = CONTRASTS[name]; col = gsea_collapsed[name].copy()
    col['pathway'] = col['Term'].map(clean)
    col = col[~col['pathway'].str.contains('|'.join(DROP_PAT), case=False, regex=True)]
    if col.empty: print(f"{name}: nothing to plot"); return
    pos = col[col.NES>0].sort_values('NES',ascending=False).head(N_SIDE)
    neg = col[col.NES<0].sort_values('NES',ascending=True).head(N_SIDE)
    d = pd.concat([neg.sort_values('NES'), pos.sort_values('NES')]).reset_index(drop=True)
    if d.empty: return
    d['lab'] = d['pathway'].map(lambda s: "\n".join(textwrap.wrap(s, width=WRAP)))
    y = np.arange(len(d))
    span = max(abs(d.NES.min()), abs(d.NES.max()))

    fig, ax = plt.subplots(figsize=(7.0, 0.46*len(d)+0.9), dpi=300)
    ax.barh(y, d['NES'], height=0.66,
            color=[UP_COL if v>0 else DN_COL for v in d['NES']],
            edgecolor='none', zorder=2)
    ax.axvline(0, color='black', lw=0.5, zorder=3)

    # label beside each bar, opposite the bar's direction
    for yi, nes, lab in zip(y, d['NES'], d['lab']):
        if nes > 0:
            ax.text(-0.03*span, yi, lab, ha='right', va='center', fontsize=8.5, linespacing=0.95)
        else:
            ax.text( 0.03*span, yi, lab, ha='left',  va='center', fontsize=8.5, linespacing=0.95)

    ax.set_yticks([]); ax.set_ylim(-0.6, len(d)-0.4)
    ax.set_xlabel("Normalized enrichment score", fontsize=9)
    ax.set_xlim(-span*1.95, span*1.95)
    # only bottom spine, hairline
    for s in ['top','right','left']: ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(left=False, labelsize=8)
    # title + minimal pole cues
    ax.set_title(cfg['axis'], fontsize=10, fontweight='bold', loc='left', pad=12)
    ax.text(0.995, 1.005, cfg['pos'], transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8.5, color=UP_COL, fontweight='bold')
    ax.text(0.005, 1.005, cfg['neg'], transform=ax.transAxes, ha='left', va='bottom',
            fontsize=8.5, color=DN_COL, fontweight='bold')

    fig.tight_layout()
    for ext in ('pdf','png','svg'):
        fig.savefig(str(OUT / f"gsea_bars_{name}.{ext}"), dpi=300, bbox_inches='tight')
    plt.show(); print(f"  saved gsea_bars_{name}.{{pdf,png,svg}}")

for name in CONTRASTS:
    if name in gsea_collapsed: gsea_bars(name)

---
## 08 · NES grids

**Why.** Pathways × contrasts as a heatmap, significance as a black dot. Where a pathway enriched in one contrast and not the others becomes legible — the separability question in enrichment form.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GSEA GRID — NES heatmap · sig = black dot · contrast labels bottom, 45°
# ════════════════════════════════════════════════════════════════════════════
import textwrap, matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import matplotlib.cm as cm
mpl.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none",
    "font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "font.size":8,"axes.linewidth":0.5})

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG, X_TAG = 'SnC', 'IRM'
FDR_SIG=0.05; N_COMMON=5; N_UNIQUE=5
DROP_PAT=['RUNX1 Interacts With Co-Factors']
FILE_TAG='4contrasts_IRM'
yp,yn,xp,xn = f'{Y_TAG}+',f'{Y_TAG}-',f'{X_TAG}+',f'{X_TAG}-'
# keys must match gsea_res / the DE tags; circular = senescence-axis contrasts
K_XpY=f'{X_TAG}axis_{Y_TAG}pos'; K_XnY=f'{X_TAG}axis_{Y_TAG}neg'
K_YpX=f'{Y_TAG}axis_{X_TAG}pos'; K_YnX=f'{Y_TAG}axis_{X_TAG}neg'
CONTRAST_ORDER=[K_XpY, K_XnY, K_YpX, K_YnX]
COL_HEADERS={
    K_XpY:f'{yp}{xp} vs {yp}{xn}',
    K_XnY:f'{yn}{xp} vs {yn}{xn}',
    K_YpX:f'{yp}{xp} vs {yn}{xp}',
    K_YnX:f'{yp}{xn} vs {yn}{xn}'}
CIRCULAR={K_YpX, K_YnX}
# ════════════════════════════════════════════════════════════════════════════
def clean(t): t=t.split('__',1)[-1]; return t.rsplit(' R-HSA',1)[0].strip()

missing=[k for k in CONTRAST_ORDER if k not in gsea_res]
if missing: print("⚠ gsea_res missing:", missing)
CONTRAST_ORDER=[k for k in CONTRAST_ORDER if k in gsea_res]

# ── pathway × contrast NES / FDR (full tables, all pathways) ────────────────
recs=[]
for name in CONTRAST_ORDER:
    df=gsea_res[name].copy(); df['pathway']=df['Term'].map(clean)
    df=df[~df['pathway'].str.contains('|'.join(DROP_PAT),case=False,regex=True)].drop_duplicates('pathway')
    for _,r in df.iterrows():
        recs.append(dict(pathway=r['pathway'],contrast=name,NES=r['NES'],FDR=r['FDR q-val']))
M=pd.DataFrame(recs)
nes=M.pivot(index='pathway',columns='contrast',values='NES').reindex(columns=CONTRAST_ORDER)
fdr=M.pivot(index='pathway',columns='contrast',values='FDR').reindex(columns=CONTRAST_ORDER)
n_sig=(fdr<FDR_SIG).sum(axis=1)

# ── rows: shared (≥3 contrasts) + contrast-specific (exactly 1) ─────────────
common=nes.loc[n_sig[n_sig>=3].index].abs().mean(axis=1).sort_values(ascending=False).head(N_COMMON).index.tolist()
blocks=[("Shared", common)]
for name in CONTRAST_ORDER:
    pool=n_sig[(n_sig==1)&(fdr[name]<FDR_SIG)].index
    top=nes.loc[pool,name].abs().sort_values(ascending=False).head(N_UNIQUE).index.tolist()
    if top: blocks.append((name, top))
row_order=[]; band=[]
for bname,terms in blocks:
    for p in sorted(terms,key=lambda x: np.nanmean(nes.loc[x])):
        if p not in row_order: row_order.append(p); band.append(bname)
row_y={p:i for i,p in enumerate(row_order)}
ncol=len(CONTRAST_ORDER); nrow=len(row_order)
vmax=np.nanmax(np.abs(nes.values)); norm=TwoSlopeNorm(vmin=-vmax,vcenter=0,vmax=vmax)
cmap=mpl.colormaps['RdBu_r']

# ── plot ────────────────────────────────────────────────────────────────────
fig,ax=plt.subplots(figsize=(6.6,0.36*nrow+2.0),dpi=300)
for p in row_order:
    y=row_y[p]
    for j,name in enumerate(CONTRAST_ORDER):
        v=nes.loc[p,name]; q=fdr.loc[p,name]
        fc=cmap(norm(v)) if not np.isnan(v) else "#f5f5f5"
        ax.add_patch(Rectangle((j-0.5,y-0.5),1,1,facecolor=fc,edgecolor='black',linewidth=1.0,zorder=2))
        if not np.isnan(q) and q<FDR_SIG:
            ax.scatter(j,y,marker='o',s=90,facecolor='black',edgecolor='black',linewidth=0.6,zorder=4)
# shade circular columns lightly so they read as "expected, not discovered"
for j,name in enumerate(CONTRAST_ORDER):
    if name in CIRCULAR:
        ax.add_patch(Rectangle((j-0.5,-0.5),1,nrow,facecolor='none',
                     edgecolor='0.4',linewidth=0.0,hatch='///',zorder=1,alpha=0.12))
# block separators
prev=None
for p,b in zip(row_order,band):
    if b!=prev and prev is not None:
        ax.axhline(row_y[p]-0.5,color='0.25',lw=0.8,zorder=5)
    prev=b
# column labels at bottom, 45°
ax.set_xticks(range(ncol))
ax.set_xticklabels([COL_HEADERS[c] for c in CONTRAST_ORDER],
                   fontsize=7.5, rotation=45, ha='right', rotation_mode='anchor')
ax.xaxis.set_ticks_position('bottom'); ax.xaxis.set_label_position('bottom')
ax.set_yticks(range(nrow))
ax.set_yticklabels(["\n".join(textwrap.wrap(p,42)) for p in row_order],fontsize=8,linespacing=0.92)
ax.set_xlim(-0.5,ncol-0.5); ax.set_ylim(nrow-0.5,-0.5)
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_aspect('equal')

# colorbar + significance legend
sm=cm.ScalarMappable(norm=norm,cmap=cmap); sm.set_array([])
cb=fig.colorbar(sm,ax=ax,fraction=0.03,pad=0.02,shrink=0.6); cb.set_label("NES",fontsize=8); cb.ax.tick_params(labelsize=7)
ax.legend(handles=[Line2D([],[],marker='o',color='w',markerfacecolor='black',
          markeredgecolor='black',markersize=9,label='FDR < 0.05')],
          loc='lower left',bbox_to_anchor=(1.02,0.0),fontsize=7,frameon=False)
ax.set_title(f"Pathway enrichment across senescence × {X_TAG} axes (microglia)",
             fontsize=9.5,fontweight='bold',loc='left',pad=10)
fig.tight_layout()
for ext in ('pdf','png','svg'):
    fig.savefig(str(OUT/f"gsea_grid_{FILE_TAG}.{ext}"),dpi=300,bbox_inches='tight')
plt.show()
print(f"grid: {nrow} × {ncol} | saved gsea_grid_{FILE_TAG}.{{pdf,png,svg}}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# COMMON vs UNIQUE — Reactome (unstripped), disease/viral artifacts removed
#   common = sig in >=3 contrasts; unique = sig in exactly 1
# ════════════════════════════════════════════════════════════════════════════
import re
PREFIX='Reactome_2022__'; FDR_SIG=0.05
# viral/disease terms = ribosomal genes mislabeled — remove (these ARE artifacts, not cherry-picking)
BLOCK=['influenza','viral','virus','infection','infected','sars','covid','corona','hiv','hcmv',
       'measles','hepatitis','tuberculosis','leishman','disease']
block_re=re.compile('|'.join(BLOCK),re.I)
def clean(t): t=t.split('__',1)[-1]; return re.sub(r'\s*R-HSA-\d+\s*$','',t).strip()

CONTRAST_ORDER=['DAMaxis_SnCpos','DAMaxis_SnCneg','SenAxis_DAMpos','SenAxis_DAMneg']
# build pathway x contrast NES/FDR
recs=[]
for name in CONTRAST_ORDER:
    df=gsea_res[name]; df=df[df['Term'].str.startswith(PREFIX)].copy()
    df['pathway']=df['Term'].map(clean)
    df=df[~df['pathway'].str.contains(block_re)].drop_duplicates('pathway')
    for _,r in df.iterrows(): recs.append(dict(pathway=r['pathway'],contrast=name,NES=r['NES'],FDR=r['FDR q-val']))
M=pd.DataFrame(recs)
nes=M.pivot(index='pathway',columns='contrast',values='NES').reindex(columns=CONTRAST_ORDER)
fdr=M.pivot(index='pathway',columns='contrast',values='FDR').reindex(columns=CONTRAST_ORDER)
n_sig=(fdr<FDR_SIG).sum(axis=1)

print(f"Reactome pathways (viral/disease removed): {len(nes)}")
print(f"  sig in >=3 contrasts (COMMON): {(n_sig>=3).sum()}")
print(f"  sig in exactly 1   (UNIQUE): {(n_sig==1).sum()}")
print(f"  sig in 2: {(n_sig==2).sum()} | never: {(n_sig==0).sum()}\n")

print("="*70)
print("COMMON (sig >=3), top 8 by mean NES:")
common=nes.loc[n_sig[n_sig>=3].index].mean(axis=1).sort_values(ascending=False).head(8)
for p,v in common.items(): print(f"  meanNES {v:+.2f}  ({n_sig[p]}/4 sig)  {p}")

print("\n"+"="*70)
print("UNIQUE (sig in exactly 1 contrast), per contrast:")
for name in CONTRAST_ORDER:
    pool=n_sig[(n_sig==1)&(fdr[name]<FDR_SIG)].index
    top=nes.loc[pool,name].abs().sort_values(ascending=False).head(5)
    print(f"\n  {name}  [{len(pool)} unique-sig total]:")
    for p in top.index: print(f"    NES {nes.loc[p,name]:+.2f}  {p}")
    if len(pool)==0: print("    (none unique to this contrast)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# REACTOME GSEA GRID — common + contrast-unique · NES heatmap · black sig dot
#   common = sig >=3 contrasts (top6 mean NES); unique = sig in exactly 1 (top4 each)
#   viral/disease artifacts removed; SLIT/ROBO kept
# ════════════════════════════════════════════════════════════════════════════
import re, textwrap, numpy as np
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import matplotlib.cm as cm
mpl.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none",
    "font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "font.size":8,"axes.linewidth":0.5})

PREFIX='Reactome_2022__'; FDR_SIG=0.05; N_COMMON=6; N_UNIQUE=4
BLOCK=['influenza','viral','virus','infection','infected','sars','covid','corona','hiv','hcmv',
       'measles','hepatitis','tuberculosis','leishman','disease']
block_re=re.compile('|'.join(BLOCK),re.I)
CONTRAST_ORDER=['DAMaxis_SnCpos','DAMaxis_SnCneg','SenAxis_DAMpos','SenAxis_DAMneg']
COL_HEADERS={'DAMaxis_SnCpos':'SnC+DAM+ vs SnC+DAM-','DAMaxis_SnCneg':'SnC-DAM+ vs SnC-DAM-',
             'SenAxis_DAMpos':'SnC+DAM+ vs SnC-DAM+','SenAxis_DAMneg':'SnC+DAM- vs SnC-DAM-'}
def clean(t): t=t.split('__',1)[-1]; return re.sub(r'\s*R-HSA-\d+\s*$','',t).strip()

# pathway x contrast NES/FDR (viral removed)
recs=[]
for name in CONTRAST_ORDER:
    df=gsea_res[name]; df=df[df['Term'].str.startswith(PREFIX)].copy()
    df['pathway']=df['Term'].map(clean)
    df=df[~df['pathway'].str.contains(block_re)].drop_duplicates('pathway')
    for _,r in df.iterrows(): recs.append(dict(pathway=r['pathway'],contrast=name,NES=r['NES'],FDR=r['FDR q-val']))
M=pd.DataFrame(recs)
nes=M.pivot(index='pathway',columns='contrast',values='NES').reindex(columns=CONTRAST_ORDER)
fdr=M.pivot(index='pathway',columns='contrast',values='FDR').reindex(columns=CONTRAST_ORDER)
n_sig=(fdr<FDR_SIG).sum(axis=1)

# selection
common=nes.loc[n_sig[n_sig>=3].index].mean(axis=1).sort_values(ascending=False).head(N_COMMON).index.tolist()
blocks=[("Shared", common)]
for name in CONTRAST_ORDER:
    pool=n_sig[(n_sig==1)&(fdr[name]<FDR_SIG)].index
    top=nes.loc[pool,name].abs().sort_values(ascending=False).head(N_UNIQUE).index.tolist()
    if top: blocks.append((name, top))

row_order=[]; band=[]
for bname,terms in blocks:
    for p in sorted(terms,key=lambda x:-np.nanmean(nes.loc[x])):
        if p not in row_order: row_order.append(p); band.append(bname)
row_y={p:i for i,p in enumerate(row_order)}
ncol=len(CONTRAST_ORDER); nrow=len(row_order)
vmax=np.nanmax(np.abs(nes.loc[row_order].values)); norm=TwoSlopeNorm(vmin=-vmax,vcenter=0,vmax=vmax)
cmap=mpl.colormaps['RdBu_r']

# block side-labels
BLABEL={'Shared':'Shared\n(translation/\nribosome)',
        'DAMaxis_SnCpos':'DAM+ |SnC+','DAMaxis_SnCneg':'DAM+ |SnC-',
        'SenAxis_DAMpos':'Sen+ |DAM+','SenAxis_DAMneg':'Sen+ |DAM-'}

fig,ax=plt.subplots(figsize=(7.0,0.40*nrow+1.8),dpi=300)
for p in row_order:
    y=row_y[p]
    for j,name in enumerate(CONTRAST_ORDER):
        v=nes.loc[p,name]; q=fdr.loc[p,name]
        fc=cmap(norm(v)) if not np.isnan(v) else "#f5f5f5"
        ax.add_patch(Rectangle((j-0.5,y-0.5),1,1,facecolor=fc,edgecolor='white',linewidth=1.0,zorder=2))
        if not np.isnan(q) and q<FDR_SIG:
            ax.scatter(j,y,marker='o',s=85,facecolor='black',edgecolor='white',linewidth=0.6,zorder=4)
# separators + right-side block labels
prev=None; brows={}
for p,b in zip(row_order,band):
    brows.setdefault(b,[]).append(row_y[p])
    if b!=prev and prev is not None: ax.axhline(row_y[p]-0.5,color='0.25',lw=0.8,zorder=5)
    prev=b
for b,ys in brows.items():
    ax.text(ncol-0.32, np.mean(ys), BLABEL.get(b,b), fontsize=6.8, fontstyle='italic',
            color='0.3', ha='left', va='center', linespacing=0.95)

ax.set_xticks(range(ncol))
ax.set_xticklabels([COL_HEADERS[c] for c in CONTRAST_ORDER],fontsize=8,rotation=45,ha='right',rotation_mode='anchor')
ax.set_yticks(range(nrow))
ax.set_yticklabels(["\n".join(textwrap.wrap(p,40)) for p in row_order],fontsize=7.5,linespacing=0.92)
ax.set_xlim(-0.5,ncol-0.5); ax.set_ylim(nrow-0.5,-0.5); ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_aspect('equal')

sm=cm.ScalarMappable(norm=norm,cmap=cmap); sm.set_array([])
cb=fig.colorbar(sm,ax=ax,fraction=0.025,pad=0.16,shrink=0.5); cb.set_label("NES",fontsize=8); cb.ax.tick_params(labelsize=7)
ax.legend(handles=[Line2D([],[],marker='o',color='w',markerfacecolor='black',markeredgecolor='white',
          markersize=8,label='FDR < 0.05')],loc='lower left',bbox_to_anchor=(1.02,-0.05),fontsize=7,frameon=False)
ax.set_title("Reactome enrichment: shared vs axis-specific programs (microglia)",
             fontsize=9.5,fontweight='bold',loc='left',pad=10)
fig.tight_layout()
OUTFIG=file.path(SCRATCH, "brain/module_06_dge/figures")
import os; os.makedirs(OUTFIG,exist_ok=True)
for ext in ('pdf','png','svg'):
    fig.savefig(f"{OUTFIG}/gsea_grid_reactome_common_unique.{ext}",dpi=300,bbox_inches='tight',facecolor='white')
plt.show()
print(f"grid: {nrow} rows x {ncol} | saved gsea_grid_reactome_common_unique.{{pdf,png,svg}}")

---
## 09 · Annotate axis-unique genes

**Why.** The genes significant on one axis only, described by function rather than left as a count.

In [ ]:
# ============================================================
# Annotate the 102 senescence-axis-unique genes by function
# ============================================================
A_uni_list = sorted(A_uni)   # the 102 unique genes (upper-case)
print(f"Total unique senescence-axis genes: {len(A_uni_list)}\n")

# manual functional buckets (literature-based gene assignments)
buckets = {
 'DNA repair / replication / DDR': ['PRKDC','XPA','FANCL','SHLD2','RAD54L2','PRIMPOL','POLA1','MMS22L','NASP','PCNP','PDK3'],
 'Chromatin / epigenetic / transcription': ['DNMT1','SUZ12','KMT2E','KAT2B','KDM1B','NSD2','RBBP4','SIN3A','TRIM24','AEBP2','MED13','MED13L','MED4','SF3B1','RBM17','SYMPK','ZBTB8OS','ZBTB44','ZNF248','ZNF407','ZNF83','STAT5B','NUCKS1','HECA'],
 'Ubiquitin / proteostasis': ['UBE2N','UBE3C','UBQLN1','UBR3','TRIP12','SMURF1','USP37','RNF145','UBAC2','PCNP'],
 'Autophagy / lysosome / vesicle': ['ATG12','WIPI1','NPC1','GGA2','ATP6V1H','PIP4P2','CLINT1','ENTPD4','SPPL3'],
 'Cytoskeleton / centrosome / cilia': ['CEP170','C2CD3','MICAL3','CCDC126','KIF5B','SPTBN1','PDLIM5','AHI1'],
 'TGFb / BMP / growth signaling': ['TGFBR2','ACVR2A','BMP2K','SMURF1','PPP2CB'],
 'Mitochondria / metabolism': ['GLUD1','MCU','MRPL3','DAP3','GLRX3','QDPR','FGGY','SFXN5','TRMT11','ZNG1B'],
 'RNA / splicing / translation-adjacent': ['NOP58','SF3B1','RBM17','PRMT3','TRMT11','DAP3'],
}

assigned = set()
for cat, genes in buckets.items():
    present = [g for g in genes if g in A_uni]
    assigned |= set(present)
    if present:
        print(f"■ {cat} ({len(present)}):")
        print(f"    {', '.join(sorted(present))}\n")

unassigned = sorted(set(A_uni_list) - assigned)
print(f"■ Unassigned / other ({len(unassigned)}):")
for i in range(0,len(unassigned),10):
    print("    "+", ".join(unassigned[i:i+10]))

# direction + effect size of the standouts
print("\n--- top unique genes by |logFC| (with direction) ---")
Au = A[A['gene'].str.upper().isin(A_uni)].copy()
Au['absLFC'] = Au['logFC'].abs()
Au = Au.sort_values('absLFC', ascending=False)
for _,r in Au.head(25).iterrows():
    print(f"  {r['gene']:12s}  logFC {r['logFC']:+.3f}  FDR {r['adj.P.Val']:.1e}  {r['direction']}")

---
## 10 · Hallmark selection

**Why.** Two blocks, by an explicit rule rather than by hand: **common** — significant in at least 3 contrasts, top N by mean NES; **unique** — significant in exactly 1, top N by |NES|. Then classified by pattern and blocked in the figure.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# HALLMARK — selection: top5 COMMON (≥3 contrasts) + top5 UNIQUE per contrast
# ════════════════════════════════════════════════════════════════════════════
# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG, X_TAG = 'SnC', 'IRM'
PREFIX='MSigDB_Hallmark_2020__'; FDR_SIG=0.05; N_COMMON=5; N_UNIQUE=5
K_XpY=f'{X_TAG}axis_{Y_TAG}pos'; K_XnY=f'{X_TAG}axis_{Y_TAG}neg'   # clean (activation axis)
K_YpX=f'{Y_TAG}axis_{X_TAG}pos'; K_YnX=f'{Y_TAG}axis_{X_TAG}neg'   # circular (senescence axis)
CONTRAST_ORDER=[K_XpY, K_XnY, K_YpX, K_YnX]
CIRCULAR={K_YpX, K_YnX}
# ════════════════════════════════════════════════════════════════════════════
def clean(t): return t.split('__',1)[-1].strip()

missing=[k for k in CONTRAST_ORDER if k not in gsea_res]
if missing: print("⚠ gsea_res missing:", missing)
CONTRAST_ORDER=[k for k in CONTRAST_ORDER if k in gsea_res]

recs=[]
for name in CONTRAST_ORDER:
    df=gsea_res[name]; df=df[df['Term'].str.startswith(PREFIX)].copy()
    df['pathway']=df['Term'].map(clean); df=df.drop_duplicates('pathway')
    for _,r in df.iterrows(): recs.append(dict(pathway=r['pathway'],contrast=name,NES=r['NES'],FDR=r['FDR q-val']))
M=pd.DataFrame(recs)
nes=M.pivot(index='pathway',columns='contrast',values='NES').reindex(columns=CONTRAST_ORDER)
fdr=M.pivot(index='pathway',columns='contrast',values='FDR').reindex(columns=CONTRAST_ORDER)
n_sig=(fdr<FDR_SIG).sum(axis=1)

# clean-only significance count (excludes the circular senescence-axis columns)
clean_cols=[c for c in CONTRAST_ORDER if c not in CIRCULAR]
n_sig_clean=(fdr[clean_cols]<FDR_SIG).sum(axis=1) if clean_cols else n_sig*0

common=nes.loc[n_sig[n_sig>=3].index].abs().mean(axis=1).sort_values(ascending=False).head(N_COMMON).index.tolist()
print(f"COMMON (sig ≥3 contrasts), top {N_COMMON}:")
for p in common:
    tag=" [both clean]" if n_sig_clean[p]==len(clean_cols) and len(clean_cols)>0 else ""
    print(f"   {p:32s} n_sig={n_sig[p]} (clean={n_sig_clean[p]})  meanNES={nes.loc[p].mean():+.2f}{tag}")
print()
for name in CONTRAST_ORDER:
    pool=n_sig[(n_sig==1)&(fdr[name]<FDR_SIG)].index
    top=nes.loc[pool,name].abs().sort_values(ascending=False).head(N_UNIQUE).index.tolist()
    flag=" (CIRCULAR — senescence-defining)" if name in CIRCULAR else ""
    print(f"UNIQUE to {name}{flag} (sig in exactly 1), top {N_UNIQUE}:  [{len(top)} found]")
    for p in top: print(f"   {nes.loc[p,name]:+.2f}  {p}")
    if not top: print("   (none — no pathway significant ONLY in this contrast)")
    print()

# ── the orthogonality read, printed explicitly ──────────────────────────────
if len(clean_cols)==2:
    both_clean=n_sig_clean[n_sig_clean==2].index
    print(f"Hallmarks sig in BOTH clean ({X_TAG}) columns [shared activation program]: {len(both_clean)}")
    for p in sorted(both_clean, key=lambda x:-abs(nes.loc[x,clean_cols].mean())):
        print(f"   {nes.loc[p,clean_cols].mean():+.2f}  {p}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# HALLMARK — top6 |NES| per contrast, union · classify by pattern · SHOW first
# ════════════════════════════════════════════════════════════════════════════
# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG, X_TAG = 'SnC', 'IRM'
PREFIX='MSigDB_Hallmark_2020__'; FDR_SIG=0.05; N_TOP=6
Xp=f'{X_TAG}axis_{Y_TAG}pos'; Xn=f'{X_TAG}axis_{Y_TAG}neg'   # activation axis | SnC+ , SnC-
Yp=f'{Y_TAG}axis_{X_TAG}pos'; Yn=f'{Y_TAG}axis_{X_TAG}neg'   # senescence axis | IRM+ , IRM-
CONTRAST_ORDER=[Xp, Xn, Yp, Yn]
# ════════════════════════════════════════════════════════════════════════════
def clean(t): return t.split('__',1)[-1].strip()

missing=[k for k in CONTRAST_ORDER if k not in gsea_res]
if missing: print("⚠ gsea_res missing:", missing)
CONTRAST_ORDER=[k for k in CONTRAST_ORDER if k in gsea_res]

# full NES/FDR matrices (all Hallmark pathways, all contrasts)
recs=[]
for name in CONTRAST_ORDER:
    df=gsea_res[name]; df=df[df['Term'].str.startswith(PREFIX)].copy()
    df['pathway']=df['Term'].map(clean); df=df.drop_duplicates('pathway')
    for _,r in df.iterrows(): recs.append(dict(pathway=r['pathway'],contrast=name,NES=r['NES'],FDR=r['FDR q-val']))
M=pd.DataFrame(recs)
nes=M.pivot(index='pathway',columns='contrast',values='NES').reindex(columns=CONTRAST_ORDER)
fdr=M.pivot(index='pathway',columns='contrast',values='FDR').reindex(columns=CONTRAST_ORDER)

# union of top-6 by |NES| per contrast (power-independent selection)
union=set()
for name in CONTRAST_ORDER:
    union |= set(nes[name].abs().sort_values(ascending=False).head(N_TOP).index)
union=sorted(union)

# classify each selected pathway by PATTERN
D1,D2,S1,S2 = Xp, Xn, Yp, Yn   # D* = activation axis, S* = senescence axis
def sig(p,c): return (not np.isnan(fdr.loc[p,c])) and fdr.loc[p,c]<FDR_SIG
def classify(p):
    d1,d2,s1,s2=nes.loc[p,D1],nes.loc[p,D2],nes.loc[p,S1],nes.loc[p,S2]
    act_up = (d1>0 and d2>0)
    # dissociation: up on activation axis, flips negative (and sig) on the senescence axis at X+
    if act_up and s1< -0.5 and (sig(p,S1) or s1<-1.5):
        return '2_dissociation'
    # senescence-specific: strongest on senescence axis (esp S2) and sig there
    if (sig(p,S2) and s2>0) or (s2>1.5 and abs(s2)>abs(d1) and abs(s2)>abs(d2)):
        return '3_sen_specific'
    # activation program: up in both activation-axis columns
    if act_up:
        return '1_act_program'
    return '4_other'
cls={p:classify(p) for p in union}
BLOCK_NAMES={'1_act_program':f'{X_TAG} program (shared by both {X_TAG}-axis contrasts)',
             '2_dissociation':'Senescence-inflammation dissociation',
             '3_sen_specific':'Senescence axis-specific',
             '4_other':'Other'}

print("="*86); print(f"HALLMARK union (top{N_TOP} |NES| per contrast) = {len(union)} pathways"); print("="*86)
for blk in ['1_act_program','2_dissociation','3_sen_specific','4_other']:
    members=[p for p in union if cls[p]==blk]
    if not members: continue
    print(f"\n▸ {BLOCK_NAMES[blk]}  ({len(members)})")
    for p in sorted(members,key=lambda x:-np.nanmean(nes.loc[x])):
        cells=" ".join(f"{nes.loc[p,c]:+.2f}{'*' if sig(p,c) else ' '}" for c in CONTRAST_ORDER)
        print(f"   {p:34s} {cells}")
print("\ncolumns order:", CONTRAST_ORDER)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# HALLMARK GSEA GRID — final · top6 |NES| union, pattern-blocked
#   drop tissue-irrelevant (Myogenesis, Spermatogenesis); NES=color, FDR<0.05=black dot
# ════════════════════════════════════════════════════════════════════════════
import textwrap, matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import matplotlib.cm as cm
mpl.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none",
    "font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "font.size":8,"axes.linewidth":0.5})

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG, X_TAG = 'SnC', 'IRM'
PREFIX='MSigDB_Hallmark_2020__'; FDR_SIG=0.05; N_TOP=6
DROP={'Myogenesis','Spermatogenesis'}
FILE_TAG='hallmark_final_IRM'
Xp=f'{X_TAG}axis_{Y_TAG}pos'; Xn=f'{X_TAG}axis_{Y_TAG}neg'   # activation axis | SnC+ , SnC-
Yp=f'{Y_TAG}axis_{X_TAG}pos'; Yn=f'{Y_TAG}axis_{X_TAG}neg'   # senescence axis | IRM+ , IRM-
CONTRAST_ORDER=[Xp, Xn, Yp, Yn]
yp,yn,xp,xn=f'{Y_TAG}+',f'{Y_TAG}-',f'{X_TAG}+',f'{X_TAG}-'
COL_HEADERS={Xp:f'{yp}{xp} vs {yp}{xn}', Xn:f'{yn}{xp} vs {yn}{xn}',
             Yp:f'{yp}{xp} vs {yn}{xp}', Yn:f'{yp}{xn} vs {yn}{xn}'}
# ════════════════════════════════════════════════════════════════════════════
def clean(t): return t.split('__',1)[-1].strip()

missing=[k for k in CONTRAST_ORDER if k not in gsea_res]
if missing: print("⚠ gsea_res missing:", missing)
CONTRAST_ORDER=[k for k in CONTRAST_ORDER if k in gsea_res]

recs=[]
for name in CONTRAST_ORDER:
    df=gsea_res[name]; df=df[df['Term'].str.startswith(PREFIX)].copy()
    df['pathway']=df['Term'].map(clean); df=df.drop_duplicates('pathway')
    for _,r in df.iterrows(): recs.append(dict(pathway=r['pathway'],contrast=name,NES=r['NES'],FDR=r['FDR q-val']))
M=pd.DataFrame(recs)
nes=M.pivot(index='pathway',columns='contrast',values='NES').reindex(columns=CONTRAST_ORDER)
fdr=M.pivot(index='pathway',columns='contrast',values='FDR').reindex(columns=CONTRAST_ORDER)
def sig(p,c): return (not np.isnan(fdr.loc[p,c])) and fdr.loc[p,c]<FDR_SIG

union=set()
for name in CONTRAST_ORDER:
    union |= set(nes[name].abs().sort_values(ascending=False).head(N_TOP).index)
union -= DROP

D1,D2,S1,S2 = Xp, Xn, Yp, Yn
def classify(p):
    d1,d2,s1,s2=nes.loc[p,D1],nes.loc[p,D2],nes.loc[p,S1],nes.loc[p,S2]
    if d1>0 and d2>0 and s1<-0.5 and (sig(p,S1) or s1<-1.5): return 1   # dissociation
    if (sig(p,S2) and s2>0) or (s2>1.5 and abs(s2)>abs(d1) and abs(s2)>abs(d2)): return 2  # sen-specific
    if d1>0 and d2>0: return 0                                          # activation program
    return 3
BLOCKS={0:f'{X_TAG} program',1:'Senescence–inflammation\ndissociation',2:'Senescence\naxis-specific',3:'Other'}
cls={p:classify(p) for p in union}

row_order=[]; band=[]
for blk in [0,1,2,3]:
    members=sorted([p for p in union if cls[p]==blk], key=lambda x:-np.nanmean(nes.loc[x]))
    for p in members: row_order.append(p); band.append(blk)
row_y={p:i for i,p in enumerate(row_order)}
ncol=len(CONTRAST_ORDER); nrow=len(row_order)
vmax=np.nanmax(np.abs(nes.loc[row_order].values)); norm=TwoSlopeNorm(vmin=-vmax,vcenter=0,vmax=vmax)
cmap=mpl.colormaps['RdBu_r']

fig,ax=plt.subplots(figsize=(6.6,0.42*nrow+1.8),dpi=300)
for p in row_order:
    y=row_y[p]
    for j,name in enumerate(CONTRAST_ORDER):
        v=nes.loc[p,name]
        fc=cmap(norm(v)) if not np.isnan(v) else "#f5f5f5"
        ax.add_patch(Rectangle((j-0.5,y-0.5),1,1,facecolor=fc,edgecolor='white',linewidth=1.2,zorder=2))
        if sig(p,name):
            ax.scatter(j,y,marker='o',s=90,facecolor='black',edgecolor='white',linewidth=0.6,zorder=4)
prev=None; block_rows={}
for p,b in zip(row_order,band):
    block_rows.setdefault(b,[]).append(row_y[p])
    if b!=prev and prev is not None: ax.axhline(row_y[p]-0.5,color='0.25',lw=0.9,zorder=5)
    prev=b
for b,ys in block_rows.items():
    ax.text(ncol-0.35, np.mean(ys), BLOCKS[b], fontsize=7.5, fontstyle='italic',
            color='0.3', ha='left', va='center', linespacing=0.95)

ax.set_xticks(range(ncol))
ax.set_xticklabels([COL_HEADERS[c] for c in CONTRAST_ORDER],fontsize=8,rotation=45,ha='right',rotation_mode='anchor')
ax.set_yticks(range(nrow)); ax.set_yticklabels(row_order,fontsize=8.5)
ax.set_xlim(-0.5,ncol-0.5); ax.set_ylim(nrow-0.5,-0.5); ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_aspect('equal')

sm=cm.ScalarMappable(norm=norm,cmap=cmap); sm.set_array([])
cb=fig.colorbar(sm,ax=ax,fraction=0.025,pad=0.18,shrink=0.55); cb.set_label("NES",fontsize=8); cb.ax.tick_params(labelsize=7)
ax.legend(handles=[Line2D([],[],marker='o',color='w',markerfacecolor='black',
          markeredgecolor='white',markersize=9,label='FDR < 0.05')],
          loc='lower left',bbox_to_anchor=(1.02,-0.08),fontsize=7,frameon=False)
ax.set_title(f"Hallmark pathway enrichment across senescence × {X_TAG} axes (microglia)",
             fontsize=9.5,fontweight='bold',loc='left',pad=10)
fig.tight_layout()
for ext in ('pdf','png','svg'):
    fig.savefig(str(OUT/f"gsea_grid_{FILE_TAG}.{ext}"),dpi=300,bbox_inches='tight')
plt.show()
print(f"Hallmark grid: {nrow} pathways × {ncol} contrasts | saved gsea_grid_{FILE_TAG}.{{pdf,png,svg}}")
print("dropped (tissue-irrelevant):", sorted(DROP))

---
## 11 · Panel K

**Why.** Colour is NES **centered per pathway**, so the grid reads as *which contrast is this pathway most enriched in* rather than *which pathway has the largest absolute NES*. Without centering a few high-NES pathways set the scale and the between-contrast differences the panel exists to show compress into one shade.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PANEL K — Row-centered GSEA grid · boxed blocks + RIGHT strip labels  (IRM)
# ════════════════════════════════════════════════════════════════════════════
import re, numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import matplotlib.cm as cm
from pathlib import Path
mpl.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none",
    "font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "font.size":8,"axes.linewidth":0.5})

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG, X_TAG = "SnC", "IRM"
OUT=Path(file.path(SCRATCH, "brain/module_06_dge/disease/AD/psychad_ad/Microglia/gsea_prerank"))
FIG=Path(file.path(SCRATCH, "brain/module_06_dge/figures")); FIG.mkdir(parents=True,exist_ok=True)
FILE_TAG="rowcentered_boxed_K_IRM"
clean=lambda t: re.sub(r'\s*R-HSA-\d+\s*$','',str(t).split('__',1)[-1]).strip()

CN=[f"{X_TAG}axis_{Y_TAG}pos", f"{X_TAG}axis_{Y_TAG}neg",
    f"{Y_TAG}axis_{X_TAG}pos", f"{Y_TAG}axis_{X_TAG}neg"]
yp,yn,xp,xn = f"{Y_TAG}+",f"{Y_TAG}-",f"{X_TAG}+",f"{X_TAG}\u2212"
COL_HEADERS=[f"{yp}{xp} vs {yp}{xn}", f"{yn}{xp} vs {yn}{xn}",
             f"{yp}{xp} vs {yn}{xp}", f"{yp}{xn} vs {yn}{xn}"]

# strings are the CLEANED display names (after clean()) — must match the CSV Term minus prefix/R-HSA
BLOCKS=[
 ("Interferon (IRM axis)","#993C1D",[
    ("Interferon Gamma Response","IFN-\u03b3 response"),
    ("Interferon Alpha Response","IFN-\u03b1 response"),
    ("Interferon Signaling","Interferon signaling"),
    ("Antiviral Mechanism By IFN-stimulated Genes","IFN-stimulated genes"),
 ]),
 ("Translation / MYC","#5F5E5A",[
    ("Eukaryotic Translation Elongation","Translation elongation"),
    ("Peptide Chain Elongation","Peptide chain elongation"),
    ("Viral mRNA Translation","Viral mRNA translation"),
    ("Myc Targets V1","MYC targets V1"),
    ("Oxidative Phosphorylation","Oxidative phosphorylation"),
 ]),
 ("Sen axis / repair","#185FA5",[
    ("DNA Repair","DNA repair"),
    ("G1/S Transition","G1/S transition"),
    ("Cellular Response To Heat Stress","Heat stress response"),
    ("Selective Autophagy","Selective autophagy"),
 ]),
 ("Inflammatory / stress","#534AB7",[
    ("TNF-alpha Signaling via NF-kB","TNF-\u03b1 / NF-\u03baB"),
    ("Regulation Of TNFR1 Signaling","TNFR1 signaling"),
    ("FOXO-mediated Transcription","FOXO transcription"),
    ("Hypoxia","Hypoxia"),
 ]),
]
# ════════════════════════════════════════════════════════════════════════════

def find_gsea(nme):
    for suffix in ("_full","_noribo","","_collapsed"):
        f=OUT/f"gsea_{nme}{suffix}.csv"
        if f.exists(): return f
    return None

res={}
for nme in CN:
    f=find_gsea(nme)
    if f is None: print(f"⚠ no GSEA file for {nme}"); continue
    print(f"  {nme:18s} ← {f.name}")
    d=pd.read_csv(f); d["p"]=d["Term"].map(clean)
    d["NES"]=pd.to_numeric(d["NES"],errors="coerce"); d["FDR q-val"]=pd.to_numeric(d["FDR q-val"],errors="coerce")
    res[nme]=d.groupby("p").first()          # collapse dup cleaned names, keep first
CN=[c for c in CN if c in res]

disp=[]; nes=[]; fdr=[]; bidx=[]
for bi,(_,_,paths) in enumerate(BLOCKS):
    for full,short in paths:
        nrow=[]; frow=[]
        for nme in CN:
            d=res[nme]
            v=d.loc[full,"NES"] if full in d.index else np.nan
            q=d.loc[full,"FDR q-val"] if full in d.index else np.nan
            nrow.append(v); frow.append(q)
        disp.append(short); nes.append(nrow); fdr.append(frow); bidx.append(bi)
NES=np.array(nes,float); FDR=np.array(fdr,float)
miss=np.isnan(NES).all(axis=1)
if miss.any(): print("⚠ not found in any contrast:", [disp[i] for i in np.where(miss)[0]])

if NES.size==0 or np.isnan(NES).all():
    raise SystemExit("No GSEA rows matched — check BLOCKS strings against the Term column (after clean()).")

# ── row-center ────────────────────────────────────────────────────────────────
NESc=NES-np.nanmean(NES,axis=1,keepdims=True)
vmax=max(np.nanmax(np.abs(NESc[~np.isnan(NESc)])),0.3); norm=TwoSlopeNorm(vmin=-vmax,vcenter=0,vmax=vmax)
cmap=mpl.colormaps["RdBu_r"]; nr,nc=NESc.shape

fig,ax=plt.subplots(figsize=(6.6,0.40*nr+1.9),dpi=300)
STRIP_W=0.55; STRIP_GAP=0.15
for i in range(nr):
    y=nr-1-i
    for j in range(nc):
        v=NESc[i,j]
        fc=cmap(norm(v)) if not np.isnan(v) else "#f5f5f5"
        ax.add_patch(Rectangle((j-0.5,y-0.5),1,1,facecolor=fc,edgecolor="white",lw=1.0,zorder=2))
        if not np.isnan(FDR[i,j]) and FDR[i,j]<0.05:
            ax.scatter(j,y,marker="o",s=70,facecolor="black",edgecolor="white",linewidth=0.6,zorder=4)

sx=nc-0.5+STRIP_GAP
for bi,(label,color,_) in enumerate(BLOCKS):
    rows=[i for i,b in enumerate(bidx) if b==bi]
    if not rows: continue
    ytop=nr-1-min(rows)+0.5; ybot=nr-1-max(rows)-0.5
    ax.add_patch(Rectangle((-0.5,ybot),nc,ytop-ybot,fill=False,edgecolor=color,lw=1.6,zorder=5))
    ax.add_patch(Rectangle((sx,ybot),STRIP_W,ytop-ybot,facecolor=color,alpha=0.16,edgecolor="none",zorder=1,clip_on=False))
    ax.add_patch(Rectangle((sx,ybot),0.06,ytop-ybot,facecolor=color,edgecolor="none",zorder=2,clip_on=False))
    ax.text(sx+STRIP_W/2,(ytop+ybot)/2,label,rotation=270,ha="center",va="center",
            fontsize=7.5,fontweight="medium",color=color,zorder=6,clip_on=False)

ax.set_xlim(-0.5, sx+STRIP_W+0.1); ax.set_ylim(-0.5,nr-0.5)
ax.set_xticks(range(nc)); ax.set_xticklabels(COL_HEADERS[:nc],fontsize=7.5,rotation=45,ha="right",rotation_mode="anchor")
ax.set_yticks(range(nr)); ax.set_yticklabels(disp[::-1],fontsize=8)
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_aspect("equal"); ax.xaxis.set_ticks_position("bottom")

sm=cm.ScalarMappable(norm=norm,cmap=cmap); sm.set_array([])
cb=fig.colorbar(sm,ax=ax,fraction=0.025,pad=0.22,shrink=0.5)
cb.set_label("NES (centered per pathway)",fontsize=7.5); cb.ax.tick_params(labelsize=7)
ax.legend(handles=[Line2D([],[],marker="o",color="w",markerfacecolor="black",markeredgecolor="white",
          markersize=8,label="FDR < 0.05")],loc="lower left",bbox_to_anchor=(1.10,-0.05),fontsize=7,frameon=False)

fig.tight_layout()
for ext in ("pdf","png","svg"):
    fig.savefig(FIG/f"gsea_grid_{FILE_TAG}.{ext}",dpi=300,bbox_inches="tight",facecolor="white")
plt.show()
print(f"✓ saved gsea_grid_{FILE_TAG}.{{pdf,png,svg}}  ({nr} rows x {nc})")
print("\nraw / centered NES per pathway (for caption):")
for i,nm in enumerate(disp):
    print(f"  {nm:26s} raw {np.array2string(NES[i],precision=2,floatmode='fixed')}  ctr {np.array2string(NESc[i],precision=2,floatmode='fixed')}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PANEL K — Row-centered GSEA grid · boxed blocks + RIGHT strip labels
#   color = NES centered per pathway (RdBu_r, TwoSlopeNorm) · black dot = FDR<0.05
#   blocks boxed + right strip labels · column headers rotated 45 (ha right)
# ════════════════════════════════════════════════════════════════════════════
import re, numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import matplotlib.cm as cm
from pathlib import Path
mpl.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none",
    "font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "font.size":8,"axes.linewidth":0.5})

OUT=Path(file.path(SCRATCH, "brain/module_06_dge/disease/AD/psychad_ad/Microglia/gsea_prerank"))
FIG=Path(file.path(SCRATCH, "brain/module_06_dge/figures")); FIG.mkdir(parents=True,exist_ok=True)
clean=lambda t: re.sub(r'\s*R-HSA-\d+\s*$','',str(t).split('__',1)[-1]).strip()

CN=["DAMaxis_SnCpos","DAMaxis_SnCneg","SenAxis_DAMpos","SenAxis_DAMneg"]
COL_HEADERS=["SnC+DAM+ vs SnC+DAM\u2212","SnC\u2212DAM+ vs SnC\u2212DAM\u2212",
             "SnC+DAM+ vs SnC\u2212DAM+","SnC+DAM\u2212 vs SnC\u2212DAM\u2212"]

# ── blocks: (label, color, [(reactome full name, display short)]) ─────────────
BLOCKS=[
 ("Shared core","#5F5E5A",[
    ("Major Pathway Of rRNA Processing In Nucleolus And Cytosol","rRNA processing"),
    ("Translation","Translation"),
    ("Cap-dependent Translation Initiation","Cap-dependent translation"),
    ("Nonsense Mediated Decay (NMD) Enhanced By Exon Junction Complex (EJC)","NMD (EJC-enhanced)"),
    ("GTP Hydrolysis And Joining Of 60S Ribosomal Subunit","60S subunit joining"),
    ("Regulation Of Expression Of SLITs And ROBOs","SLIT/ROBO expression"),
 ]),
 ("DAM axis","#993C1D",[
    ("Senescence-Associated Secretory Phenotype (SASP)","SASP"),
    ("Iron Uptake And Transport","Iron uptake & transport"),
    ("Protein Ubiquitination","Protein ubiquitination"),
    ("E3 Ubiquitin Ligases Ubiquitinate Target Proteins","E3 ubiquitin ligases"),
    ("Innate Immune System","Innate immune system"),
 ]),
 ("Sen axis (DAM+)","#185FA5",[
    ("Gap-filling DNA Repair Synthesis And Ligation In TC-NER","TC-NER gap-filling"),
    ("Dual Incision In TC-NER","TC-NER dual incision"),
    ("Formation Of TC-NER Pre-Incision Complex","TC-NER pre-incision"),
 ]),
 ("Sen axis (DAM\u2212)","#534AB7",[
    ("Cell Cycle Checkpoints","Cell cycle checkpoints"),
    ("Global Genome Nucleotide Excision Repair (GG-NER)","GG-NER"),
    ("Selective Autophagy","Selective autophagy"),
 ]),
]

# ── reload GSEA, build NES/FDR matrices ───────────────────────────────────────
res={}
for nme in CN:
    f=OUT/f"gsea_{nme}_full.csv"
    if not f.exists(): f=OUT/f"gsea_{nme}_noribo.csv"
    d=pd.read_csv(f); d["p"]=d["Term"].map(clean)
    d["NES"]=pd.to_numeric(d["NES"],errors="coerce"); d["FDR q-val"]=pd.to_numeric(d["FDR q-val"],errors="coerce")
    res[nme]=d.set_index("p")

disp=[]; nes=[]; fdr=[]; bidx=[]
for bi,(_,_,paths) in enumerate(BLOCKS):
    for full,short in paths:
        nrow=[]; frow=[]
        for nme in CN:
            d=res[nme]
            v=d.loc[full,"NES"] if full in d.index else np.nan
            q=d.loc[full,"FDR q-val"] if full in d.index else np.nan
            if isinstance(v,pd.Series): v=v.iloc[0]
            if isinstance(q,pd.Series): q=q.iloc[0]
            nrow.append(v); frow.append(q)
        disp.append(short); nes.append(nrow); fdr.append(frow); bidx.append(bi)
NES=np.array(nes,float); FDR=np.array(fdr,float)
miss=np.isnan(NES).all(axis=1)
if miss.any(): print("⚠ not found:", [disp[i] for i in np.where(miss)[0]])

# ── row-center ────────────────────────────────────────────────────────────────
NESc=NES-np.nanmean(NES,axis=1,keepdims=True)
vmax=max(np.nanmax(np.abs(NESc)),0.3); norm=TwoSlopeNorm(vmin=-vmax,vcenter=0,vmax=vmax)
cmap=mpl.colormaps["RdBu_r"]
nr,nc=NESc.shape

fig,ax=plt.subplots(figsize=(6.6,0.40*nr+1.9),dpi=300)
STRIP_W=0.55                       # strip width in data units (cols = 1 unit)
STRIP_GAP=0.15                     # gap between grid and strip

# cells (row 0 at top)
for i in range(nr):
    y=nr-1-i
    for j in range(nc):
        v=NESc[i,j]
        fc=cmap(norm(v)) if not np.isnan(v) else "#f5f5f5"
        ax.add_patch(Rectangle((j-0.5,y-0.5),1,1,facecolor=fc,edgecolor="white",lw=1.0,zorder=2))
        if not np.isnan(FDR[i,j]) and FDR[i,j]<0.05:
            ax.scatter(j,y,marker="o",s=70,facecolor="black",edgecolor="white",linewidth=0.6,zorder=4)

# block boxes + RIGHT strip labels
sx=nc-0.5+STRIP_GAP                 # strip starts right of the last column
for bi,(label,color,_) in enumerate(BLOCKS):
    rows=[i for i,b in enumerate(bidx) if b==bi]
    ytop=nr-1-min(rows)+0.5; ybot=nr-1-max(rows)-0.5
    ax.add_patch(Rectangle((-0.5,ybot),nc,ytop-ybot,fill=False,edgecolor=color,lw=1.6,zorder=5))
    ax.add_patch(Rectangle((sx,ybot),STRIP_W,ytop-ybot,facecolor=color,alpha=0.16,edgecolor="none",zorder=1,clip_on=False))
    ax.add_patch(Rectangle((sx,ybot),0.06,ytop-ybot,facecolor=color,edgecolor="none",zorder=2,clip_on=False))
    ax.text(sx+STRIP_W/2,(ytop+ybot)/2,label,rotation=270,ha="center",va="center",
            fontsize=7.5,fontweight="medium",color=color,zorder=6,clip_on=False)

# axes
ax.set_xlim(-0.5, sx+STRIP_W+0.1); ax.set_ylim(-0.5,nr-0.5)
ax.set_xticks(range(nc))
ax.set_xticklabels(COL_HEADERS,fontsize=7.5,rotation=45,ha="right",rotation_mode="anchor")
ax.set_yticks(range(nr)); ax.set_yticklabels(disp[::-1],fontsize=8)
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_aspect("equal")
ax.xaxis.set_ticks_position("bottom")

# colorbar + FDR legend (pad pushed out so it clears the right strip)
sm=cm.ScalarMappable(norm=norm,cmap=cmap); sm.set_array([])
cb=fig.colorbar(sm,ax=ax,fraction=0.025,pad=0.22,shrink=0.5)
cb.set_label("NES (centered per pathway)",fontsize=7.5); cb.ax.tick_params(labelsize=7)
ax.legend(handles=[Line2D([],[],marker="o",color="w",markerfacecolor="black",markeredgecolor="white",
          markersize=8,label="FDR < 0.05")],loc="lower left",bbox_to_anchor=(1.10,-0.05),fontsize=7,frameon=False)

fig.tight_layout()
for ext in ("pdf","png","svg"):
    fig.savefig(FIG/f"gsea_grid_rowcentered_boxed_K.{ext}",dpi=300,bbox_inches="tight",facecolor="white")
plt.show()
print(f"✓ saved gsea_grid_rowcentered_boxed_K.{{pdf,png,svg}}  ({nr} rows x {nc})")
print("\nraw / centered NES per pathway (for caption):")
for i,nm in enumerate(disp):
    print(f"  {nm:26s} raw {np.array2string(NES[i],precision=2,floatmode='fixed')}  ctr {np.array2string(NESc[i],precision=2,floatmode='fixed')}")